In [ ]:
"""
Viscous flux Jacobian at wall boundary faces for 3D Navier-Stokes.

The boundary Jacobian follows the same chain-rule pattern as the interior
Jacobian, but the "right" state is a ghost state U_b = f(U_i, BC):

    J_wall = J_i_interior + J_j_interior @ dUb_dUi

where J_i_interior, J_j_interior come from the same viscous_flux_jacobians()
used at interior faces, and dUb_dUi = d(U_b)/d(U_i) is the BC Jacobian.
"""

import numpy as np
import sympy as sp


# -----------------------------------------------------------------------
# Ghost state construction  (numerical, for flux evaluation)
# -----------------------------------------------------------------------

def get_ghost_state(U_i, n, bc_type, T_wall=300.0, gamma=1.4, Rgas=287.0):
    """
    Build the ghost conservative state U_b for a boundary face.

    Parameters
    ----------
    U_i      : (5,) interior conservative state [rho, rhou, rhov, rhow, rhoE]
    n        : (3,) outward unit face normal (pointing away from interior)
    bc_type  : 'slip_wall' | 'no_slip_adiabatic' | 'no_slip_isothermal'
    T_wall   : wall temperature (only used for 'no_slip_isothermal')

    Returns
    -------
    U_b : (5,) ghost conservative state
    """
    rho, u, v, w, p, T = cons_to_prim(U_i, gamma, Rgas)
    nx, ny, nz = n

    if bc_type == 'slip_wall':
        # Reflect only the normal velocity component (Euler wall, no viscosity)
        un    = u*nx + v*ny + w*nz
        u_b   = u - 2.0*un*nx
        v_b   = v - 2.0*un*ny
        w_b   = w - 2.0*un*nz
        T_b   = T
        p_b   = p

    elif bc_type == 'no_slip_adiabatic':
        # Full velocity reflection -> u_face = 0
        # Mirror temperature      -> T_b = T_i  =>  dT/dn = 0  (adiabatic)
        u_b, v_b, w_b = -u, -v, -w
        T_b = T          # same T -> zero normal gradient
        p_b = p          # zero normal pressure gradient

    elif bc_type == 'no_slip_isothermal':
        # Full velocity reflection -> u_face = 0
        # Ghost T chosen so that (T_i + T_b)/2 = T_wall
        u_b, v_b, w_b = -u, -v, -w
        T_b = 2.0*T_wall - T
        p_b = p          # zero normal pressure gradient

    else:
        raise ValueError(f"Unknown bc_type: {bc_type}")

    rho_b = p_b / (Rgas * T_b)
    E_b   = p_b / ((gamma - 1.0)*rho_b) + 0.5*(u_b**2 + v_b**2 + w_b**2)
    return np.array([rho_b, rho_b*u_b, rho_b*v_b, rho_b*w_b, rho_b*E_b])


# -----------------------------------------------------------------------
# BC Jacobian  dU_b / dU_i  via sympy  (same approach as your interior code)
# -----------------------------------------------------------------------

def get_ghost_jacobian(U_i, n, bc_type, T_wall=300.0, gamma=1.4, Rgas=287.0):
    """
    Compute dU_b/dU_i (5x5) by differentiating the ghost state construction
    symbolically w.r.t. the interior conservative variables.

    This is the 'duRduL' matrix in the Fortran code.
    """
    nx, ny, nz = n

    # Symbolic interior conservative variables
    rho_s, rhou_s, rhov_s, rhow_s, rhoE_s = sp.symbols(
        'rho rhou rhov rhow rhoE', positive=True
    )
    U_s = sp.Matrix([rho_s, rhou_s, rhov_s, rhow_s, rhoE_s])

    # Derived primitive variables (symbolic)
    u_s = rhou_s / rho_s
    v_s = rhov_s / rho_s
    w_s = rhow_s / rho_s
    E_s = rhoE_s / rho_s
    p_s = (gamma - 1) * rho_s * (E_s - sp.Rational(1,2)*(u_s**2+v_s**2+w_s**2))
    T_s = p_s / (rho_s * Rgas)

    # Ghost primitive variables (symbolic functions of U_s)
    if bc_type == 'slip_wall':
        un_s  = u_s*nx + v_s*ny + w_s*nz
        ub_s  = u_s - 2*un_s*nx
        vb_s  = v_s - 2*un_s*ny
        wb_s  = w_s - 2*un_s*nz
        Tb_s  = T_s
        pb_s  = p_s

    elif bc_type == 'no_slip_adiabatic':
        ub_s, vb_s, wb_s = -u_s, -v_s, -w_s
        Tb_s = T_s
        pb_s = p_s

    elif bc_type == 'no_slip_isothermal':
        ub_s, vb_s, wb_s = -u_s, -v_s, -w_s
        Tb_s = 2*T_wall - T_s
        pb_s = p_s

    # Ghost conservative variables (symbolic)
    rhob_s  = pb_s / (Rgas * Tb_s)
    Eb_s    = pb_s / ((gamma-1)*rhob_s) + sp.Rational(1,2)*(ub_s**2+vb_s**2+wb_s**2)
    Ub_s    = sp.Matrix([
        rhob_s,
        rhob_s * ub_s,
        rhob_s * vb_s,
        rhob_s * wb_s,
        rhob_s * Eb_s
    ])

    # Differentiate ghost state w.r.t. interior conservative state
    dUb_dUi_sym = Ub_s.jacobian(U_s)

    # Evaluate at the actual interior state
    rho, u, v, w, p, T = cons_to_prim(U_i, gamma, Rgas)
    E = U_i[4] / rho
    subs = {
        rho_s: rho, rhou_s: U_i[1], rhov_s: U_i[2],
        rhow_s: U_i[3], rhoE_s: U_i[4]
    }
    dUb_dUi_num = np.array(dUb_dUi_sym.subs(subs)).astype(np.float64)
    return dUb_dUi_num


# -----------------------------------------------------------------------
# Wall boundary viscous flux Jacobian  (main entry point)
# -----------------------------------------------------------------------

def viscous_boundary_flux_jacobian(U_i, n, ds, bc_type,
                                    T_wall=300.0,
                                    gamma=1.4, Rgas=287.0, Pr=0.72,
                                    mu0=1.716e-5, T0=273.15, Suth_C=110.4):
    """
    Compute d(F_n^v)/d(U_i) at a boundary face.

    The ghost state U_b is constructed from U_i via the chosen BC,
    then the chain rule gives:

        J_wall = J_i + J_j @ dUb_dUi

    where J_i, J_j are from viscous_flux_jacobians(U_i, U_b, n, ds),
    and dUb_dUi = d(U_b)/d(U_i) from the BC construction.

    Parameters
    ----------
    U_i     : (5,) interior conservative state
    n       : (3,) outward unit face normal
    ds      : distance from interior cell centre to wall face
              NOTE: for a ghost-cell approach, ds here is typically
              2 * (cell-centre to wall distance), consistent with
              treating the ghost as a mirror image.
    bc_type : 'slip_wall' | 'no_slip_adiabatic' | 'no_slip_isothermal'
    T_wall  : wall temperature (isothermal BC only)

    Returns
    -------
    J_wall : (5,5)  d(F_n^v)/d(U_i)  -- goes into diagonal block only.
    """
    U_b = get_ghost_state(U_i, n, bc_type, T_wall, gamma, Rgas)

    # Flux Jacobians treating (U_i, U_b) like an interior face
    J_i, J_j = viscous_flux_jacobians(U_i, U_b, n, ds,
                                       gamma, Rgas, Pr, mu0, T0, Suth_C)

    # BC Jacobian: how ghost state responds to interior state
    dUb_dUi = get_ghost_jacobian(U_i, n, bc_type, T_wall, gamma, Rgas)

    # Chain rule: total Jacobian wrt U_i
    J_wall = J_i + J_j @ dUb_dUi
    return J_wall


# -----------------------------------------------------------------------
# Finite-difference verification
# -----------------------------------------------------------------------

def viscous_boundary_flux_jacobian_fd(U_i, n, ds, bc_type, eps=1e-6,
                                       T_wall=300.0,
                                       gamma=1.4, Rgas=287.0, Pr=0.72,
                                       mu0=1.716e-5, T0=273.15, Suth_C=110.4):
    """FD check: perturb U_i, rebuild ghost, evaluate flux, finite difference."""
    def flux(U):
        U_b = get_ghost_state(U, n, bc_type, T_wall, gamma, Rgas)
        return Fnv_numeric(U, U_b, n, ds, gamma, Rgas, Pr, mu0, T0, Suth_C)

    F0 = flux(U_i)
    J  = np.zeros((5, 5))
    for k in range(5):
        U_pert    = U_i.copy()
        U_pert[k] += eps
        J[:, k]   = (flux(U_pert) - F0) / eps
    return J


# -----------------------------------------------------------------------
# Example / sanity check
# -----------------------------------------------------------------------
if __name__ == "__main__":
    np.set_printoptions(precision=4, suppress=False, linewidth=120)

    gamma, Rgas = 1.4, 287.0

    def prim_to_cons(rho, u, v, w, p):
        E = p / ((gamma-1)*rho) + 0.5*(u**2+v**2+w**2)
        return np.array([rho, rho*u, rho*v, rho*w, rho*E])

    U_i    = prim_to_cons(1.20, 50.0, 5.0, 1.0, 101325.0)
    n      = np.array([1.0, 0.0, 0.0])   # normal pointing into domain boundary
    ds     = 0.01                         # 2 * (cell-centre to wall distance)
    T_wall = 300.0

    for bc in ['slip_wall', 'no_slip_adiabatic', 'no_slip_isothermal']:
        print(f"\n{'='*60}")
        print(f"BC type: {bc}")
        print(f"{'='*60}")

        J_analytic = viscous_boundary_flux_jacobian(
            U_i, n, ds, bc, T_wall=T_wall, gamma=gamma, Rgas=Rgas)
        J_fd       = viscous_boundary_flux_jacobian_fd(
            U_i, n, ds, bc, eps=1e-6, T_wall=T_wall, gamma=gamma, Rgas=Rgas)

        print("Analytic J_wall:")
        print(J_analytic)
        print("\nFinite-diff J_wall:")
        print(J_fd)
        print(f"\nmax |error| = {np.max(np.abs(J_analytic - J_fd)):.3e}")

ModuleNotFoundError: No module named 'viscous_flux_jacobian'